## condition concepts

In [0]:
--CREATE VASCULAR DX TABLE
--THIS REPLICATES THE ORIGINAL COHORT DX INCLUSION CRITERIA
--THIS ESSENTIALLY TELLS US WHY THE PT IS INCLUDED IN THIS DATASET

drop table if exists nctracs.vascular_dx;
create table nctracs.vascular_dx as

select ROW_NUMBER() OVER(PARTITION BY deid_person_id ORDER BY condition_start_date) as row_number, x.* from (

select * from nctracs.vascular_condition
where condition_source_value in ('I73.89','I73.9','I77.70','I77.72','I77.76','I77.77','I77.79','I79.8') 
or substr(condition_source_value, 1,5) in ('E08.5','E09.5','E10.5','E11.5','E13.5')
or substr(condition_source_value, 1,3) in ('I70','I74','I75','I76','I96','L97')
and substr(condition_source_value,1,3) not in ('I71','I72') and condition_source_value not in ('I70.1')
order by deid_person_id, condition_start_date
) x

In [0]:
--DEPRESSIVE DISORDER
CREATE OR REPLACE TEMP VIEW depression as
select distinct deid_person_id, count(distinct condition_start_date) as freq_depres from (
select b.concept_name, a.* 
from nctracs.vascular_condition a LEFT JOIN nctracs.vascular_concept b on a.condition_concept_id = b.CONCEPT_ID
where a.condition_concept_id IN (select distinct descendant_concept_id from nctracs.vascular_concept_ancestors where ancestor_concept_id=440383)
) X group by deid_person_id

## measurement concepts

In [0]:
--DIALYSIS FROM OBSERVATIONS
--create data to identify apparent first dialysis service identified

select concept_name, a.*
from nctracs.vascular_observation a LEFT JOIN nctracs.vascular_concept b on a.observation_concept_id = b.CONCEPT_ID
where observation_concept_id = '4019967'


In [0]:
--A1c LABS
--filter out impossible a1c values that are over 100 (in theory top end of a1c is around 20)

CREATE OR REPLACE TEMP VIEW diabetes as
select distinct deid_person_id, 
        min(MEASUREMENT_DATE) as a1c_first, 
        max(measurement_date) as a1c_last, 
        count(measurement_id) as a1c_freq,
        min(VALUE_AS_NUMBER) as a1c_min,
        avg(VALUE_AS_NUMBER) as a1c_mean,
        max(VALUE_AS_NUMBER) as a1c_max
        from nctracs.vascular_measurement 
        where measurement_concept_id in (3004410)
        and value_as_number<100
        group by deid_person_id

In [0]:
select * from nctracs.vascular_measurement where measurement_concept_id = 3004410
and value_as_number>50

In [0]:
-- LIPOPROTEIN A

CREATE OR REPLACE TEMP VIEW lpa as
select deid_person_id, 
count(measurement_id) as freq_lpa,
avg(VALUE_AS_NUMBER) as avg_lpa from nctracs.vascular_measurement 
where MEASUREMENT_CONCEPT_ID in (4007663,46284886,4197857,3013861,3046664)
and VALUE_AS_NUMBER is not null
group by DEID_PERSON_ID

In [0]:
--GFR

CREATE OR REPLACE TEMP VIEW gfr as
with gfr_values as(
select * from nctracs.vascular_measurement
where MEASUREMENT_CONCEPT_ID in (select distinct concept_id from nctracs.vascular_concept_freqs where lower(concept_name) like ('%glomerular filtration%') and domain='measurement' )
)
select distinct deid_person_id, 
                count(measurement_id) as gfr_freq,
                min(value_as_number) as gfr_min,
                max(value_as_number) as gfr_max,
                avg(value_as_number) as gfr_mean
from gfr_values
group by DEID_PERSON_ID

In [0]:
--ALBUMIN LABS
CREATE OR REPLACE TEMP VIEW albumin as
select distinct deid_person_id, 
        min(MEASUREMENT_DATE) as alb_first, 
        max(measurement_date) as alb_last, 
        count(measurement_id) as alb_freq,
        min(VALUE_AS_NUMBER) as alb_min,
        avg(VALUE_AS_NUMBER) as alb_mean,
        max(VALUE_AS_NUMBER) as alb_max
        from nctracs.vascular_measurement where measurement_concept_id in (3024561)
        group by deid_person_id

In [0]:
--LDL labs

--ancestor option A  4012479:  LDL cholesterol measurement -> gives 453,693 rows
--ancestor option B 4331302:  LDL measurement -> gives 454,687 - this gives weird nmol values

CREATE OR REPLACE TEMP VIEW ldl as
WITH LDL as(
SELECT DISTINCT d.DEID_PERSON_ID, c.concept_name, d.*
FROM  nctracs.vascular_measurement d  
JOIN nctracs.vascular_concept c ON d.measurement_CONCEPT_ID = c.concept_id
JOIN nctracs_environmental.concept_ancestor ca ON d.measurement_concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id = '4012479'
) 
select deid_person_id, 
      count(measurement_id) as ldl_freq,
      min(value_as_number) as ldl_min,
      max(value_as_number) as ldl_max,
      avg(value_as_number) as ldl_mean
      from LDL 
      group by deid_person_id

## drug concepts

In [0]:
--ANTITHROMBOTICS

CREATE OR REPLACE TEMP VIEW thromb as
with antithromb as (
  select * from nctracs.vascular_drug_exposure where drug_concept_id in (select distinct descendant_concept_id from nctracs.vascular_concept_ancestors 
where ancestor_concept_id in 21600960)
)

select deid_person_id, count(drug_exposure_id) as freq_antithromb
 from antithromb
 group by deid_person_id


In [0]:
--STATINS

CREATE OR REPLACE TEMP VIEW statin as
with statins as (
SELECT de.*
FROM nctracs.vascular_drug_exposure de
JOIN nctracs_environmental.concept_ancestor ca ON de.drug_concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id IN (
  1510813, 1545958, 1539403, 1551860, 1592085, 1549686, 40165636)
)
select deid_person_id, min(drug_exposure_start_date) as min_statin_date,
max(drug_exposure_start_date) as max_statin_date,
count(drug_exposure_id) as freq_statin_drug
from statins
group by deid_person_id

In [0]:
--ANTIHYPERTENSIVES

CREATE OR REPLACE TEMP VIEW htn as
with hyper_meds as(
select b.concept_name, a.* 
from nctracs.vascular_drug_exposure a LEFT JOIN nctracs.vascular_concept b on a.drug_concept_id = b.CONCEPT_ID
where a.drug_concept_id IN (select distinct descendant_concept_id from nctracs.vascular_concept_ancestors where ancestor_concept_id=21600381)
)
select deid_person_id, min(drug_exposure_start_date) as min_htn_drug,
 max(drug_exposure_start_date) as max_htn_drug,
 count(drug_exposure_id) as freq_htn_drug
from hyper_meds
group by deid_person_id

In [0]:
--ANTIDIABETES DRUGS

CREATE OR REPLACE TEMP VIEW DIAB_MED AS
with diab_meds as(
select b.concept_name, a.* 
from nctracs.vascular_drug_exposure a LEFT JOIN nctracs.vascular_concept b on a.drug_concept_id = b.CONCEPT_ID
where a.drug_concept_id IN (select distinct descendant_concept_id from nctracs.vascular_concept_ancestors where ancestor_concept_id=21600712)
)
select deid_person_id, min(drug_exposure_start_date) as min_diab_drug, 
max(drug_exposure_end_date) as max_diab_drug,
count(drug_exposure_id) as freq_diab_drug
from diab_meds
group by deid_person_Id

## OBSERVATION CONCEPTS

In [0]:
--financial for food (or housing, medical care, heating)
--excluded are clear negative responses
--included are positive or choose not to respond or don't know

CREATE OR REPLACE TEMP VIEW sdoh as
with finance as(
select b.concept_name, a.* from nctracs.vascular_observation a LEFT JOIN nctracs.vascular_concept b on a.observation_concept_id = b.CONCEPT_ID
where observation_concept_id in (37116643,36306143,36304041,46234789,4010022)
and VALUE_AS_CONCEPT_ID not in (36309869,21498606,1621261)
AND VALUE_AS_STRING is not null
)
select distinct deid_person_id, count(observation_id) as freq_finance
from finance
group by deid_person_id

In [0]:
--active cigarette smoker or passive smoker

CREATE OR REPLACE TEMP VIEW smoke as
select deid_person_id, 
min(observation_date) as min_smoking,
max(observation_date) as max_smoking,
count(observation_id) as freq_smoking
 from nctracs.vascular_observation
where observation_concept_id in (903656, 903657) 
group by deid_person_id

In [0]:
--TRANSPORTATION ISSUES
--excludes response of 'no'
--includes both affirmative and choose not to respond

CREATE OR REPLACE TEMP VIEW trans as
with transpo as(
select b.concept_name, a.* from nctracs.vascular_observation a LEFT JOIN nctracs.vascular_concept b on a.observation_concept_id = b.CONCEPT_ID
where observation_concept_id in (3964826, 37020730)
and value_as_string !='No'
)
select deid_person_id, count(observation_id) as freq_transpo
from transpo
group by deid_person_id

## procedure concepts

In [0]:
--CREATE TABLE OF REVASCULARIZATIONS & CREATE MARKERS TO SUMMARIZE PROCS

CREATE OR REPLACE TEMP VIEW procs as
with revasc as(
select 
case when lower(concept_name) like ('%dilation%') then 1 else 0 end as dilation,
case when lower(concept_name) like ('%bypass%') then 1 else 0 end as bypass,
case when lower(concept_name) like ('%extirpation%') then 1 else 0 end as extirpation,
case when lower(concept_name) like ('%arterectomy%') then 1 else 0 end as arterectomy,
case when lower(concept_name) like ('%reoperation%') then 1 else 0 end as reoperation,
case when lower(concept_name) like ('%thrombectomy%') then 1 else 0 end as thrombectomy,
concept_name, 
a.*

from nctracs.vascular_procedure_occurrence a LEFT JOIN nctracs.vascular_concept b on a.procedure_concept_id = b.CONCEPT_ID

where PROCEDURE_SOURCE_VALUE in (
34201,34203,35302,35303,35304,35305,35306,35331,35351,35355,35361,35363,35371,35372,
35537,35637,35538,35638,35539,35647,35540,35646,35556,35656,35558,35661,35563,35663,
35565,35665,35566,35666,35570,35571,35671,35583,35585,35587,35621,35623,35654,35700,
35875,35879,35881,35883,35884)

or PROCEDURE_SOURCE_VALUE in (
  '04CK0ZZ',	'04CL0ZZ',	'35656',	'35371',	'34201',	'35666',	'047L3ZZ',	'04CL3ZZ',	'041L0JL',	'35661',	'047K3ZZ',	'04CK3ZZ',	'35372',	'041K0JL',	'047Q3ZZ',	'35302',	'047P3ZZ',	'34203',	'047N3ZZ',	'35875',	'047R3ZZ',	'047U3ZZ',	'047M3ZZ',	'047T3ZZ',	'047S3ZZ',	'04CM0ZZ',	'35700',	'04CN0ZZ',	'35556',	'35646',	'04CM3ZZ',	'047C3DZ',	'047D3DZ',	'04CN3ZZ',	'047K3DZ',	'047L3DZ',	'047H3DZ',	'35876',	'35566',	'047J3DZ',	'047D3ZZ',	'047C3ZZ',	'047Y3ZZ',	'04CH0ZZ',	'041L09L',	'04CJ0ZZ',	'047J3ZZ',	'35654',	'047L34Z',	'35583',	'04CY0ZZ',	'047H3ZZ',	'041K09L',	'041K0JJ',	'04100JK',	'35585',	'04CP3ZZ',	'047K34Z',	'04CR3ZZ',	'04CS0ZZ',	'041L0JH',	'04CC0ZZ',	'35571',	'04CR0ZZ',	'041L0JN',	'04CU3ZZ',	'04CP0ZZ',	'047N34Z',	'047M34Z',	'04CQ3ZZ',	'04CT3ZZ',	'04CS3ZZ',	'04CD0ZZ',	'04CQ0ZZ',	'047M3DZ',	'35351',	'047N3DZ',	'35303',	'04703DZ',	'04CU0ZZ',	'35883',	'041K0JN',	'047K0ZZ',	'35665',	'047D34Z',	'041K09N',	'04CT0ZZ',	'047H0DZ',	'067D3DZ',	'047C34Z',	'04C00ZZ',	'35621',	'041K0ZL',	'047L0ZZ',	'047C0DZ',	'047P34Z',	'04CH3ZZ',	'047J34Z',	'047D0DZ',	'047H34Z',	'047Q34Z',	'041L09N',	'047J0DZ',	'03743DZ',	'047S34Z',	'041K0JM',	'04CJ3ZZ',	'04CD3ZZ',	'04CC3ZZ',	'041K0JQ',	'041L0JM',	'35671',	'04CY3ZZ',	'041N09Q',	'041K0JH',	'047P3DZ',	'041L0KL',	'03150J8',	'35638',	'047U34Z',	'047K0DZ',	'041L0ZL',	'047T34Z',	'041K0KL',	'047V3ZZ',	'041L0JJ',	'047Y3DZ',	'35304',	'047H0ZZ',	'041K0KN',	'041L09M',	'03160J7',	'041M09Q',	'057Y3DZ',	'03150J6',	'047N0ZZ',	'35331',	'041L0KN',	'047P0ZZ',	'35305',	'047R34Z',	'047M0ZZ',	'04BL0ZZ',	'047C0ZZ',	'04BK0ZZ',	'04100J8',	'047W3ZZ',	'041L0ZH',	'047C04Z',	'35558',	'067F3DZ',	'35587',	'047Y34Z',	'047D0ZZ',	'03150J9',	'047K04Z',	'041K09M',	'04LL3DZ',	'047H04Z',	'35355',	'047L0DZ',	'041L09Q',	'041K0ZJ',	'03160JC',	'047T3DZ',	'047J04Z',	'047J0ZZ',	'047L04Z',	'041K09Q',	'041K0JK',	'04LK3ZZ',	'047S3DZ',	'04703ZZ',	'047S0ZZ',	'04LK0ZZ',	'041L0ZN',	'04104J8',	'047E3ZZ',	'04100JJ',	'35647',	'35623',	'04100ZK',	'047M0DZ',	'047R0ZZ',	'04CK4ZZ',	'047D04Z',	'04C03ZZ',	'041K0ZN',	'047Y0ZZ',	'041M09M',	'047T0ZZ',	'047Q0ZZ',	'041C0JH',	'047Q3DZ',	'047U3DZ',	'047F3ZZ',	'041N0JQ',	'047U0ZZ',	'041H0JH',	'047F34Z',	'041L0AH',	'041J0JJ',	'04BM0ZZ',	'041M09P',	'041L0JQ',	'04700DZ',	'041J0JH',	'04CW3ZZ',	'041L4JL',	'047E34Z',	'03160JB',	'04BY0ZZ',	'047R3DZ',	'041M0JQ',	'35637',	'04CM4ZZ',	'04LL3ZZ',	'041L0JS',	'04CV3ZZ',	'041M09L',	'041D0JJ',	'041M0JL',	'047E3DZ',	'04LK3DZ',	'04LL0ZZ',	'041K0ZM',	'041L0KH',	'35540',	'04CE0ZZ',	'04CW0ZZ',	'047C4DZ',	'061M09Y',	'041K4ZL',	'041N09P',	'041L0ZJ',	'04100JG',	'35570',	'04CE3ZZ',	'04CR4ZZ',	'041L0AL',	'047N0DZ',	'047M04Z',	'35565',	'045Y0ZZ',	'047C44Z',	'04100JH',	'047N4ZZ',	'041L0KM',	'047N04Z',	'03160Z7',	'041K0KS',	'041K49L',	'04CN4ZZ',	'041K49N',	'041M0JM',	'047J44Z',	'35879',	'047E0DZ',	'03150A8',	'041L09P',	'047D4ZZ',	'047M4ZZ',	'04CT4ZZ',	'04BY4ZZ',	'047K44Z',	'047034Z',	'041M0KS',	'041K0KM',	'041K0ZS',	'047P04Z',	'047F0DZ',	'047D4DZ',	'041N0JL',	'04CV0ZZ',	'041L4JH',	'041N09M',	'041L4KL',	'041L09S',	'041K0JS',	'041L0KJ',	'047Q4ZZ',	'04CP4ZZ',	'041K4JJ',	'047S4ZZ',	'04LN0ZZ',	'047C4ZZ',	'041L0JK',	'041N09L',	'03150JC',	'041K0KK',	'047Y0DZ',	'04LU0ZZ',	'041K0KJ',	'047T4ZZ',	'041N0KQ',	'35361',	'041K0KQ',	'04LQ0ZZ',	'041L4AL',	'04BP0ZZ',	'04100KH',	'04100ZF',	'041L0ZK',	'04RK07Z',	'041K4JM',	'03160A8',	'04LK0CZ',	'047T0DZ',	'04BN0ZZ',	'041L4JJ',	'041K0AK',	'047U0DZ',	'041K0AL',	'03150Z6',	'047N4DZ',	'041L0ZS',	'04100JR',	'041K0ZH',	'041N0ZS',	'041K09J',	'04BS0ZZ',	'041L4ZH',	'041F0JJ',	'041K09P',	'041L0AJ',	'041L0KQ',	'04BR0ZZ',	'047J4DZ',	'047L4ZZ',	'04LR3DZ',	'041M0KQ',	'041N0KP',	'35663',	'04CF0ZZ',	'04100J7',	'04100JQ',	'041K4JL',	'047E0ZZ',	'047U4ZZ',	'041L0KS',	'041K4JK',	'047R44Z',	'03150AC',	'04LM3ZZ',	'03150J7',	'04CF3ZZ',	'047D44Z',	'35884',	'04RK0JZ',	'047K4ZZ',	'03160A7',	'04CU4ZZ',	'04100JF',	'041K0AJ',	'04LL0CZ',	'047H4ZZ',	'041K0AN',	'35306',	'041K4KN',	'04RK0KZ',	'041E0JH',	'04104JC',	'041K4ZM',	'047P0DZ',	'03150A6',	'04RL0KZ',	'047H4DZ',	'047Q0DZ',	'047N44Z',	'041N0JP',	'041K09H',	'04104KD',	'04100J6',	'041K0KH',	'04LK0DZ',	'04LP0ZZ',	'04100Z8',	'04LS3ZZ',	'041K4JN',	'04104ZK',	'041N0ZM',	'35881',	'04100ZD',	'047L4DZ',	'041L49L',	'047E04Z',	'061G0JY',	'041D0JH',	'041L0AN',	'04LR3ZZ',	'041M0ZQ',	'04100AK',	'041N09S',	'041N0JM',	'047J4ZZ',	'041K0ZK',	'047H44Z'
)

order by DEID_PERSON_ID, PROCEDURE_DATE
)

select distinct DEID_PERSON_ID, 
min(PROCEDURE_DATE) as min_revasc_date, 
max(PROCEDURE_DATE) as max_revasc_date, 
count(distinct PROCEDURE_DATE) as proc_days,
sum(dilation) as dilation,
sum(bypass) as bypass,
sum(extirpation) as extirpation,
sum(arterectomy) as arterectomy,
sum(reoperation) as reoperation,
sum(thrombectomy) as thrombectomy
from revasc
group by DEID_PERSON_ID

In [0]:
--CREATE TABLE FOR AMPUTATIONS DATA
--these gives counts of codes, that's why a person can have multiple amputations on same day or very close in time

CREATE OR REPLACE TEMP VIEW amp as
with amps as (
select 
case when procedure_concept_id in (4159766,40480578,2105806,4143797,4272232,4302020) then 1 else 0 end as amp_toe,
case when procedure_concept_id in (4054983,4264289,2105451,2006242) then 1 else 0 end as amp_foot,
case when procedure_concept_id in (2105448,4195136,4338257,2105223,2105222,2101638,2006243,4143795,2105450,2105449,36675618,2105211,2105447) then 1 else 0 end as amp_leg,
case when procedure_concept_id in (4078404) then 1 else 0 end as amp_revision,
concept_name,
 a.*

from nctracs.vascular_procedure_occurrence a LEFT JOIN nctracs.vascular_concept b on a.PROCEDURE_CONCEPT_ID = b.concept_id 
where procedure_concept_id in (

4345680,42536891,4176170,4308713,4103642,4202322,133088,135722,196613,4101660,4103640,4103638,4113102,2105806,4272232,4338257,4195136,4302020,4143797,2105450,4159766,2101638,2105448,2105223,2006243,2105449,4054983,4143795,4264289,2105211,4078404,2105222,40480578,2105447,2105451,2006242,36675618) 
order by DEID_PERSON_ID, PROCEDURE_DATE
) 

select distinct DEID_PERSON_ID, 
        min(PROCEDURE_DATE) as min_amp_date, 
        max(PROCEDURE_DATE) as max_amp_date, 
        sum(amp_toe) as amp_toe,
        sum(amp_foot) as amp_foot,
        sum(amp_leg) as amp_leg,
        sum(amp_revision) as amp_revision
        from amps 
        group by DEID_PERSON_ID

In [0]:
select 
p.deid_person_id,
floor(date_diff(current_date(), p.birth_datetime)/365.25) as age,
p.race_concept_id,
p.ethnicity_concept_id,
d.death_date, 
d.cod1,
dep.freq_depres,
diabetes.a1c_first,
diabetes.a1c_last,
diabetes.a1c_freq,
diabetes.a1c_min,
diabetes.a1c_mean,
diabetes.a1c_max,
lpa.freq_lpa,
lpa.avg_lpa,
gfr.gfr_freq,
gfr.gfr_min,
gfr.gfr_mean,
gfr.gfr_max,
albumin.alb_freq,
albumin.alb_min,
albumin.alb_mean,
albumin.alb_max,
ldl.ldl_freq,
ldl.ldl_min,
ldl.ldl_mean,
ldl.ldl_max,
thromb.freq_antithromb,
statin.min_statin_date,
statin.max_statin_date,
statin.freq_statin_drug,
htn.min_htn_drug,
htn.max_htn_drug,
htn.freq_htn_drug,
diab_med.min_diab_drug,
diab_med.max_diab_drug,
diab_med.freq_diab_drug,
sdoh.freq_finance,
smoke.min_smoking,
smoke.max_smoking,
smoke.freq_smoking,
trans.freq_transpo,
procs.min_revasc_date,
procs.max_revasc_date,
procs.dilation,
procs.bypass,
procs.extirpation,
procs.arterectomy,
procs.reoperation,
procs.thrombectomy,
amp.min_amp_date,
amp.max_amp_date,
amp.amp_toe,
amp.amp_foot,
amp.amp_leg,
amp.amp_revision
from nctracs.vascular_person p LEFT JOIN nctracs.vascular_state_death_data d on p.deid_person_id=d.deid_person_id
                               LEFT JOIN depression dep on p.deid_person_id = dep.deid_person_id
                               LEFT JOIN diabetes on p.deid_person_id = diabetes.deid_person_id
                               LEFT JOIN lpa on p.DEID_PERSON_ID = lpa.deid_person_id
                               LEFT JOIN gfr on p.DEID_PERSON_ID = gfr.deid_person_id
                               LEFT JOIN albumin on p.DEID_PERSON_ID = albumin.deid_person_id
                               LEFT JOIN ldl on p.DEID_PERSON_ID = ldl.deid_person_id
                               LEFT JOIN thromb on p.DEID_PERSON_ID = thromb.deid_person_id
                               LEFT JOIN statin on p.DEID_PERSON_ID = statin.deid_person_id
                               LEFT JOIN htn on p.DEID_PERSON_ID = htn.deid_person_id
                               LEFT JOIN diab_med on p.DEID_PERSON_ID = diab_med.deid_person_id
                               LEFT JOIN sdoh on p.DEID_PERSON_ID = sdoh.deid_person_id
                               LEFT JOIN smoke on p.DEID_PERSON_ID = smoke.deid_person_id
                               LEFT JOIN trans on p.DEID_PERSON_ID = trans.deid_person_id
                               LEFT JOIN procs on p.DEID_PERSON_ID = procs.deid_person_id
                               LEFT JOIN amp on p.DEID_PERSON_ID = amp.deid_person_id

